# Cloning the repository

In [1]:
%cd /kaggle/working

!rm -rf /kaggle/working/Autoformer

!git clone \
    --branch experiment2 \
    --single-branch \
    https://github.com/faribaghorbani/Autoformer.git

%cd /kaggle/working/Autoformer

/kaggle/working
Cloning into 'Autoformer'...
remote: Enumerating objects: 404, done.
remote: Counting objects: 100% (287/287), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 404 (delta 214), reused 206 (delta 195), pack-reused 117 (from 1)
Receiving objects: 100% (404/404), 2.22 MiB | 20.46 MiB/s, done.
Resolving deltas: 100% (240/240), done.
/kaggle/working/Autoformer


In [2]:
!git branch --show-current
!git status
!git log -1 --oneline

commit = !git rev-parse HEAD
print("Experiment2 commit:", commit[0])

experiment2
On branch experiment2
Your branch is up to date with 'origin/experiment2'.

nothing to commit, working tree clean
970e7d0 (HEAD -> experiment2, origin/experiment2) update autoformer.py
Experiment2 commit: 970e7d02aca3107ad6bdfdcd1a353081cdbbb3f8


## install dependencies

In [3]:
!pip install -q -r requirements.txt

In [4]:
import reformer_pytorch
print("reformer_pytorch imported successfully")

reformer_pytorch imported successfully


# Check the files 

In [5]:
!grep -n "period_mode\|router_hidden\|router_bins" run.py

55:    '--period_mode',
63:        '--router_hidden',
70:        '--router_bins',
136:                                                                                                                  args.period_mode,
137:                                                                                                                  args.router_hidden,
138:                                                                                                                  args.router_bins,
171:                                                                                                      args.period_mode,
172:                                                                                                      args.router_hidden,
173:                                                                                                      args.router_bins,


In [6]:
!grep -n "period_mode\|router_hidden\|router_bins" models/Autoformer.py

21:        period_mode = getattr(
23:            'period_mode',
27:        router_hidden = getattr(
29:            'router_hidden',
33:        router_bins = getattr(
35:            'router_bins',
61:                                period_mode=period_mode,
62:                                router_hidden=router_hidden,
63:                                router_bins=router_bins
85:                            period_mode=period_mode,
86:                            router_hidden=router_hidden,
87:                            router_bins=router_bins
96:                            period_mode=period_mode,
97:                            router_hidden=router_hidden,
98:                            router_bins=router_bins


In [7]:
!grep -n "time_delay_agg_samplewise\|time_delay_agg_router\|period_mode" layers/AutoCorrelation.py

23:    period_mode='original',
35:        self.period_mode = period_mode
39:        if self.period_mode == 'router':
102:    def time_delay_agg_samplewise(self, values, corr):
207:    def time_delay_agg_router(self, values, corr):
459:        if self.period_mode == 'original':
473:        elif self.period_mode == 'samplewise':
476:            V = self.time_delay_agg_samplewise(
481:        elif self.period_mode == 'router':
484:            V = self.time_delay_agg_router(
491:                f"Unknown period_mode: {self.period_mode}"


In [8]:
!sed -n '50,78p' run.py

    parser.add_argument('--moving_avg', type=int, default=25, help='window size of moving average')
    parser.add_argument('--factor', type=int, default=1, help='attn factor')

    # experiment 2A
    parser.add_argument(
    '--period_mode',
    type=str,
    default='original',
    choices=['original', 'samplewise', 'router'],
    help='Auto-Correlation period selection mode'
    )

    parser.add_argument(
        '--router_hidden',
        type=int,
        default=16,
        help='hidden dimension of adaptive period router'
    )

    parser.add_argument(
        '--router_bins',
        type=int,
        default=8,
        help='number of pooled autocorrelation bins used by period router'
    )
    parser.add_argument('--distil', action='store_false',
                        help='whether to use distilling in encoder, using this argument means not using distilling',
                        default=True)
    parser.add_argument('--dropout', type=float, default=0.05, help='dropou

In [9]:
!python run.py --help | grep -A2 period_mode

2026-08-13:21:38:33,739 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
              [--factor FACTOR] [--period_mode {original,samplewise,router}]
              [--router_hidden ROUTER_HIDDEN] [--router_bins ROUTER_BINS]
              [--distil] [--dropout DROPOUT] [--embed EMBED]
--
  --period_mode {original,samplewise,router}
                        Auto-Correlation period selection mode
  --router_hidden ROUTER_HIDDEN


In [10]:
!python -m py_compile \
    run.py \
    models/Autoformer.py \
    layers/AutoCorrelation.py

In [11]:
!python run.py --help | grep -E "period_mode|router_hidden|router_bins"

2026-08-13:21:39:14,192 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
              [--factor FACTOR] [--period_mode {original,samplewise,router}]
              [--router_hidden ROUTER_HIDDEN] [--router_bins ROUTER_BINS]
  --period_mode {original,samplewise,router}
  --router_hidden ROUTER_HIDDEN
  --router_bins ROUTER_BINS


In [12]:
import sys
import torch

sys.path.insert(0, "/kaggle/working/Autoformer")

from layers.AutoCorrelation import AutoCorrelation

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

B = 4
L = 96
H = 8
E = 64

torch.manual_seed(123)

q = torch.randn(B, L, H, E, device=device)
k = torch.randn(B, L, H, E, device=device)
v = torch.randn(B, L, H, E, device=device)

Device: cuda


In [13]:
original = AutoCorrelation(
    mask_flag=False,
    factor=1,
    attention_dropout=0.0,
    output_attention=False,
    period_mode="original",
    router_hidden=16,
    router_bins=8,
).to(device)

samplewise = AutoCorrelation(
    mask_flag=False,
    factor=1,
    attention_dropout=0.0,
    output_attention=False,
    period_mode="samplewise",
    router_hidden=16,
    router_bins=8,
).to(device)

router = AutoCorrelation(
    mask_flag=False,
    factor=1,
    attention_dropout=0.0,
    output_attention=False,
    period_mode="router",
    router_hidden=16,
    router_bins=8,
).to(device)

In [14]:
samplewise.train()
router.train()

out_2a, _ = samplewise(q, k, v, None)
out_2b, _ = router(q, k, v, None)

print("Input shape :", q.shape)
print("2A shape    :", out_2a.shape)
print("2B shape    :", out_2b.shape)

assert out_2a.shape == q.shape
assert out_2b.shape == q.shape

print("✓ Output shapes are correct")

Input shape : torch.Size([4, 96, 8, 64])
2A shape    : torch.Size([4, 96, 8, 64])
2B shape    : torch.Size([4, 96, 8, 64])
✓ Output shapes are correct


## correctness of 2A

In [16]:
original.eval()
samplewise.eval()

with torch.no_grad():

    out_original_eval, _ = original(
        q, k, v, None
    )

    out_2a_eval, _ = samplewise(
        q, k, v, None
    )

diff = (
    out_original_eval
    - out_2a_eval
).abs().max().item()

print(
    "Maximum original-vs-2A inference difference:",
    diff
)

Maximum original-vs-2A inference difference: 0.0


In [17]:
assert torch.allclose(
    out_original_eval,
    out_2a_eval,
    atol=1e-5,
    rtol=1e-5
)

print(
    "✓ 2A inference matches original inference"
)

✓ 2A inference matches original inference


In [18]:
original.train()
samplewise.train()

with torch.no_grad():

    out_original_train, _ = original(
        q, k, v, None
    )

    out_2a_train, _ = samplewise(
        q, k, v, None
    )

train_diff = (
    out_original_train
    - out_2a_train
).abs().mean().item()

print(
    "Mean original-vs-2A training difference:",
    train_diff
)

Mean original-vs-2A training difference: 0.4403880536556244


## correctness of 2B

In [19]:
samplewise.train()
router.train()

with torch.no_grad():

    out_2a_initial, _ = samplewise(
        q, k, v, None
    )

    out_2b_initial, _ = router(
        q, k, v, None
    )

router_initial_diff = (
    out_2a_initial
    - out_2b_initial
).abs().max().item()

print(
    "Initial 2A-vs-2B difference:",
    router_initial_diff
)

Initial 2A-vs-2B difference: 0.0


In [20]:
assert torch.allclose(
    out_2a_initial,
    out_2b_initial,
    atol=1e-5,
    rtol=1e-5
)

print(
    "✓ Router correctly starts from 2A behavior"
)

✓ Router correctly starts from 2A behavior


## check the learning ability of neural net

In [21]:
router.train()
router.zero_grad()

out, _ = router(
    q, k, v, None
)

loss = out.pow(2).mean()

loss.backward()

print("Router gradients:")

for name, param in router.named_parameters():

    if "period_router" in name:

        if param.grad is None:
            print(name, "-> NO GRADIENT")

        else:
            print(
                name,
                "->",
                param.grad.abs().mean().item()
            )

Router gradients:
period_router.0.weight -> 0.0
period_router.0.bias -> 0.0
period_router.2.weight -> 0.000426074315328151
period_router.2.bias -> 1.30385160446167e-08


### defining the rout and path

In [15]:
import os

DATASET_ROOT = (
    "/kaggle/input/datasets/"
    "faribaghorbani/autoformer-dataset/dataset"
)

ETTm2_ROOT = os.path.join(
    DATASET_ROOT,
    "ETT-small"
)

# test the run.py file and arguments

In [16]:
!python -u run.py \
    --is_training 1 \
    --root_path "$ETTm2_ROOT" \
    --data_path ETTm2.csv \
    --model_id ETTm2_smoke_2A \
    --model Autoformer \
    --data ETTm2 \
    --features M \
    --seq_len 96 \
    --label_len 48 \
    --pred_len 96 \
    --e_layers 2 \
    --d_layers 1 \
    --factor 1 \
    --enc_in 7 \
    --dec_in 7 \
    --c_out 7 \
    --d_model 512 \
    --batch_size 32 \
    --learning_rate 0.0001 \
    --train_epochs 1 \
    --patience 1 \
    --period_mode samplewise \
    --des smoke_2A \
    --itr 1

2026-08-13:21:42:00,408 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_smoke_2A', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=96, bucket_size=4, n_hashes=4, enc_in=7, dec_in=7, c_out=7, d_model=512, n_heads=8, e_layers=2, d_layers=1, d_ff=2048, moving_avg=25, factor=1, period_mode='samplewise', router_hidden=16, router_bins=8, distil=True, dropout=0.05, embed='timeF', activation='gelu', output_attention=False, do_predict=False, num_workers=10, itr=1, train_epochs=1, batch_size=32, patience=1, learning_rate=0.0001, des='smoke_2A', loss='mse', lradj='type1', use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3')
Use GPU: cuda:0
>>>>>>>start training : ETTm2_smoke_2A_Autoformer_ETTm2_ftM_sl96_ll48_p

In [25]:
!python -u run.py \
    --is_training 1 \
    --root_path "$ETTm2_ROOT" \
    --data_path ETTm2.csv \
    --model_id ETTm2_smoke_2B \
    --model Autoformer \
    --data ETTm2 \
    --features M \
    --seq_len 96 \
    --label_len 48 \
    --pred_len 96 \
    --e_layers 2 \
    --d_layers 1 \
    --factor 1 \
    --enc_in 7 \
    --dec_in 7 \
    --c_out 7 \
    --d_model 512 \
    --batch_size 32 \
    --learning_rate 0.0001 \
    --train_epochs 1 \
    --patience 1 \
    --period_mode router \
    --router_hidden 16 \
    --router_bins 8 \
    --des smoke_2B \
    --itr 1

2026-08-13:18:49:48,172 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_smoke_2B', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=96, bucket_size=4, n_hashes=4, enc_in=7, dec_in=7, c_out=7, d_model=512, n_heads=8, e_layers=2, d_layers=1, d_ff=2048, moving_avg=25, factor=1, period_mode='router', router_hidden=16, router_bins=8, distil=True, dropout=0.05, embed='timeF', activation='gelu', output_attention=False, do_predict=False, num_workers=10, itr=1, train_epochs=1, batch_size=32, patience=1, learning_rate=0.0001, des='smoke_2B', loss='mse', lradj='type1', use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3')
Use GPU: cuda:0
>>>>>>>start training : ETTm2_smoke_2B_Autoformer_ETTm2_ftM_sl96_ll48_pl96_

## Create a separate Experiment 2 results file

In [17]:
import os
import pandas as pd

RESULTS_FILE = "/kaggle/working/autoformer_experiment2_results.csv"

if not os.path.exists(RESULTS_FILE):

    columns = [
        "dataset",
        "period_mode",
        "root_path",
        "data_path",
        "features",
        "seq_len",
        "label_len",
        "pred_len",
        "enc_in",
        "dec_in",
        "c_out",
        "e_layers",
        "d_layers",
        "factor",
        "d_model",
        "batch_size",
        "learning_rate",
        "train_epochs",
        "patience",
        "router_hidden",
        "router_bins",
        "mse",
        "mae",
        "status",
        "notes",
    ]

    pd.DataFrame(columns=columns).to_csv(
        RESULTS_FILE,
        index=False
    )

print("Results file:")
print(RESULTS_FILE)

print("\nCurrent contents:")
display(pd.read_csv(RESULTS_FILE))

Results file:
/kaggle/working/autoformer_experiment2_results.csv

Current contents:


,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes


### Experiment runner function

In [18]:
import os
import re
import subprocess
import pandas as pd
from datetime import datetime

REPO = "/kaggle/working/Autoformer"
RESULTS_FILE = "/kaggle/working/autoformer_experiment2_results.csv"


def run_autoformer_experiment(
    dataset,
    root_path,
    data_path,
    data_type,
    seq_len,
    label_len,
    pred_len,
    enc_in,
    dec_in,
    c_out,
    period_mode,
    train_epochs=10,
    patience=3,
    factor=1,
    batch_size=32,
    learning_rate=1e-4,
    router_hidden=16,
    router_bins=8,
):

    # ---------------------------------------------------------
    # Give every mode its own model/checkpoint name
    # ---------------------------------------------------------
    model_id = (
        f"{dataset}_{seq_len}_{pred_len}_{period_mode}"
    )

    cmd = [
        "python", "-u", "run.py",

        "--is_training", "1",

        "--root_path", root_path,
        "--data_path", data_path,

        "--model_id", model_id,
        "--model", "Autoformer",
        "--data", data_type,

        "--features", "M",

        "--seq_len", str(seq_len),
        "--label_len", str(label_len),
        "--pred_len", str(pred_len),

        "--e_layers", "2",
        "--d_layers", "1",

        "--factor", str(factor),

        "--enc_in", str(enc_in),
        "--dec_in", str(dec_in),
        "--c_out", str(c_out),

        "--d_model", "512",

        "--batch_size", str(batch_size),
        "--learning_rate", str(learning_rate),

        "--train_epochs", str(train_epochs),
        "--patience", str(patience),

        # -----------------------------------------------------
        # Experiment 2 configuration
        # -----------------------------------------------------
        "--period_mode", period_mode,

        "--router_hidden", str(router_hidden),
        "--router_bins", str(router_bins),

        "--des", f"experiment2_{period_mode}",

        "--itr", "1",
    ]

    print("=" * 100)
    print(f"DATASET: {dataset}")
    print(f"INPUT LENGTH: {seq_len}")
    print(f"PREDICTION LENGTH: {pred_len}")
    print(f"PERIOD MODE: {period_mode}")
    print("=" * 100)

    print("\nCommand:")
    print(" ".join(cmd))

    print("\n" + "=" * 100)
    print("LIVE TRAINING LOG")
    print("=" * 100)

    start = datetime.now()

    # ---------------------------------------------------------
    # Run process and stream output live
    # ---------------------------------------------------------
    process = subprocess.Popen(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []

    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
        output_lines.append(line)

    process.stdout.close()

    return_code = process.wait()

    output = "".join(output_lines)

    # ---------------------------------------------------------
    # Extract final MSE and MAE
    # ---------------------------------------------------------
    matches = re.findall(
        r"mse:([0-9eE.+-]+), mae:([0-9eE.+-]+)",
        output
    )

    mse = None
    mae = None

    if matches:
        mse = float(matches[-1][0])
        mae = float(matches[-1][1])

    status = (
        "success"
        if return_code == 0 and mse is not None
        else "failed"
    )

    # ---------------------------------------------------------
    # Save result
    # ---------------------------------------------------------
    row = {
        "dataset": dataset,
        "period_mode": period_mode,
        "root_path": root_path,
        "data_path": data_path,
        "features": "M",
        "seq_len": seq_len,
        "label_len": label_len,
        "pred_len": pred_len,
        "enc_in": enc_in,
        "dec_in": dec_in,
        "c_out": c_out,
        "e_layers": 2,
        "d_layers": 1,
        "factor": factor,
        "d_model": 512,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "train_epochs": train_epochs,
        "patience": patience,
        "router_hidden": router_hidden,
        "router_bins": router_bins,
        "mse": mse,
        "mae": mae,
        "status": status,
        "notes": str(start),
    }

    result_df = pd.DataFrame([row])

    result_df.to_csv(
        RESULTS_FILE,
        mode="a",
        header=False,
        index=False,
    )

    print("\n" + "=" * 100)
    print("EXPERIMENT FINISHED")
    print("=" * 100)

    print(f"Mode: {period_mode}")
    print(f"Return code: {return_code}")
    print(f"MSE: {mse}")
    print(f"MAE: {mae}")
    print(f"Status: {status}")

    print("\nResult appended to:")
    print(RESULTS_FILE)

    display(result_df)

    return result_df

## define the dataset path

In [19]:
DATASET_ROOT = (
    "/kaggle/input/datasets/"
    "faribaghorbani/autoformer-dataset/dataset"
)

ETTm2_ROOT = os.path.join(
    DATASET_ROOT,
    "ETT-small"
)

print(ETTm2_ROOT)
print(
    "ETTm2 exists:",
    os.path.exists(
        os.path.join(
            ETTm2_ROOT,
            "ETTm2.csv"
        )
    )
)

/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small
ETTm2 exists: True


## Experiment 2A only on horizon 96

In [29]:
result_2a_96 = run_autoformer_experiment(

    dataset="ETTm2",

    root_path=ETTm2_ROOT,
    data_path="ETTm2.csv",

    data_type="ETTm2",

    seq_len=96,
    label_len=48,
    pred_len=96,

    enc_in=7,
    dec_in=7,
    c_out=7,

    period_mode="samplewise",

    train_epochs=10,
    patience=3,

    factor=1,
    batch_size=32,
    learning_rate=1e-4,
)

DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 96
PERIOD MODE: samplewise

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_96_samplewise --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 96 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode samplewise --router_hidden 16 --router_bins 8 --des experiment2_samplewise --itr 1

LIVE TRAINING LOG
2026-08-13:18:58:51,282 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_96_samplewise', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,samplewise,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,96,7,7,...,32,0.0001,10,3,16,8,0.221951,0.307343,success,2026-08-13 18:58:49.405898


## experiment 2A + 2B on horizon 96

In [30]:
result_router_96 = run_autoformer_experiment(

    dataset="ETTm2",

    root_path=ETTm2_ROOT,
    data_path="ETTm2.csv",

    data_type="ETTm2",

    seq_len=96,
    label_len=48,
    pred_len=96,

    enc_in=7,
    dec_in=7,
    c_out=7,

    period_mode="router",

    router_hidden=16,
    router_bins=8,

    train_epochs=10,
    patience=3,

    factor=1,
    batch_size=32,
    learning_rate=1e-4,
)

DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 96
PERIOD MODE: router

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_96_router --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 96 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode router --router_hidden 16 --router_bins 8 --des experiment2_router --itr 1

LIVE TRAINING LOG
2026-08-13:19:17:58,068 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_96_router', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, p

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,router,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,96,7,7,...,32,0.0001,10,3,16,8,0.236419,0.317191,success,2026-08-13 19:17:56.209605


## Eperiment 2A on all horizons

In [32]:
for pred_len in [
    96,
    192,
    336,
    720
]:

    run_autoformer_experiment(

        dataset="ETTm2",

        root_path=ETTm2_ROOT,
        data_path="ETTm2.csv",

        data_type="ETTm2",

        seq_len=96,
        label_len=48,
        pred_len=pred_len,

        enc_in=7,
        dec_in=7,
        c_out=7,

        # ==========================================
        # EXPERIMENT 2A
        # ==========================================
        period_mode="samplewise",

        train_epochs=10,
        patience=3,

        factor=1,
        batch_size=32,
        learning_rate=1e-4,
    )

DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 96
PERIOD MODE: samplewise

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_96_samplewise --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 96 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode samplewise --router_hidden 16 --router_bins 8 --des experiment2_samplewise --itr 1

LIVE TRAINING LOG
2026-08-13:19:37:10,337 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_96_samplewise', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,samplewise,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,96,7,7,...,32,0.0001,10,3,16,8,0.255995,0.323098,success,2026-08-13 19:37:08.387573


DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 192
PERIOD MODE: samplewise

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_192_samplewise --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 192 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode samplewise --router_hidden 16 --router_bins 8 --des experiment2_samplewise --itr 1

LIVE TRAINING LOG
2026-08-13:19:55:45,082 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_192_samplewise', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,samplewise,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,192,7,7,...,32,0.0001,10,3,16,8,0.280576,0.340491,success,2026-08-13 19:55:43.109182


DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 336
PERIOD MODE: samplewise

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_336_samplewise --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 336 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode samplewise --router_hidden 16 --router_bins 8 --des experiment2_samplewise --itr 1

LIVE TRAINING LOG
2026-08-13:20:20:15,304 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_336_samplewise', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,samplewise,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,336,7,7,...,32,0.0001,10,3,16,8,0.331455,0.366371,success,2026-08-13 20:20:13.277906


DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 720
PERIOD MODE: samplewise

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_720_samplewise --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 720 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode samplewise --router_hidden 16 --router_bins 8 --des experiment2_samplewise --itr 1

LIVE TRAINING LOG
2026-08-13:20:42:40,829 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_720_samplewise', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,samplewise,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,720,7,7,...,32,0.0001,10,3,16,8,0.425174,0.419483,success,2026-08-13 20:42:38.854826


## Experiment 2A + 2B on all horizons

In [20]:
for pred_len in [
    96,
    192,
    336,
    720
]:

    run_autoformer_experiment(

        dataset="ETTm2",

        root_path=ETTm2_ROOT,
        data_path="ETTm2.csv",

        data_type="ETTm2",

        seq_len=96,
        label_len=48,
        pred_len=pred_len,

        enc_in=7,
        dec_in=7,
        c_out=7,

        # ==========================================
        # EXPERIMENT 2A + 2B
        # ==========================================
        period_mode="router",

        router_hidden=16,
        router_bins=8,

        train_epochs=10,
        patience=3,

        factor=1,
        batch_size=32,
        learning_rate=1e-4,
    )

DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 96
PERIOD MODE: router

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_96_router --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 96 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode router --router_hidden 16 --router_bins 8 --des experiment2_router --itr 1

LIVE TRAINING LOG
2026-08-13:21:46:51,795 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_96_router', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, p

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,router,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,96,7,7,...,32,0.0001,10,3,16,8,0.232079,0.313985,success,2026-08-13 21:46:49.816435


DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 192
PERIOD MODE: router

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_192_router --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 192 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode router --router_hidden 16 --router_bins 8 --des experiment2_router --itr 1

LIVE TRAINING LOG
2026-08-13:21:59:41,882 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_192_router', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=4

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,router,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,192,7,7,...,32,0.0001,10,3,16,8,0.272445,0.330676,success,2026-08-13 21:59:39.673497


DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 336
PERIOD MODE: router

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_336_router --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 336 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode router --router_hidden 16 --router_bins 8 --des experiment2_router --itr 1

LIVE TRAINING LOG
2026-08-13:22:16:52,547 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_336_router', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=4

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,router,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,336,7,7,...,32,0.0001,10,3,16,8,0.32328,0.360564,success,2026-08-13 22:16:50.501062


DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 720
PERIOD MODE: router

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_720_router --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 720 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --period_mode router --router_hidden 16 --router_bins 8 --des experiment2_router --itr 1

LIVE TRAINING LOG
2026-08-13:22:46:04,096 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_720_router', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=4

,dataset,period_mode,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,...,batch_size,learning_rate,train_epochs,patience,router_hidden,router_bins,mse,mae,status,notes
0,ETTm2,router,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,720,7,7,...,32,0.0001,10,3,16,8,0.423647,0.418412,success,2026-08-13 22:46:01.666631
